# Projet TSA

### Imports

In [284]:
import numpy as np
import cv2

from scipy import signal
from scipy.fftpack import fft
from scipy.linalg import toeplitz

import pandas as pd
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import struct

import msicpe
print(msicpe.__version__)
from msicpe import tsa

1.0.21


*On rappelle que la documentation de la bibliothèque msicpe peut être trouvée en ligne --> __[msicpe](https://cpe.pages.in2p3.fr/msi/toolbox/msicpe.tsa.html)__.*

## Séance 1 - MDFB

### Chargement des données


In [285]:
Fs, s = msicpe.tsa.load_signal('audio')

# réduction de la dimension de s
s = s[:1500]

In [286]:
# propriétés du signal
N = 1500            # nombre de points du signal s
D = N/Fs            # durée du signal s

# vecteur temps
ts = np.linspace(0, (N-1)/Fs, N)

In [287]:
# affichage du signal s
Npts_to_plot = N

df_s = pd.DataFrame({'x': ts[:Npts_to_plot], 'y': s[:Npts_to_plot], 'legende':'Signal audio'})
fig = px.line(df_s, x='x',  y='y', color='legende', labels={'x':'Temps (s)', 'y':''}, title='Signal temporel', width=700)
fig.show()

### Binarisation

In [288]:
# signal binarisé
dtype = 'int'
sb = msicpe.tsa.data2bin(s, dtype=dtype)
Nb = N               # nombre de points du signal sb

# vecteur temps correspondant
d_bin = 10      # frequence d'echantillonnage du signal binaire
tb = np.linspace(0, Nb/d_bin, Nb, endpoint=False)

## **ATTENTION**
L’affichage d’un signal possédant un très grand nombre de points peut faire crasher VSCode.
Aussi, pour chaque signal que vous souhaiterez afficher, pensez à **réduire sa dimension** **dans la dataframe créée pour l’affichage**.

In [289]:
# Affichage de s_b 
Nbits_to_plot = 120

df_sb = pd.DataFrame({'x': tb[:Nbits_to_plot], 'y': sb[:Nbits_to_plot], 'legende':'sb'})
fig2 = px.line(df_sb,x='x',  y='y', color='legende', markers="*", labels={'x': 'Temps (s)' , 'y':'Signal Binarise'}, title= 'Binarisation du Signal' , width=700 )
fig2.show()

### Modulation

In [290]:
# paramètres de modulation
A   = 5
nu0 = 20
nu1 = 40

# vecteur des instants d’échantillonnage du signal modulé s_m
F_mod = 1000
T_mod  = 1/F_mod
N_mod  = int(F_mod/d_bin)

tm_bit = np.linspace(0, N_mod / F_mod, N_mod, endpoint=False)


# signal modulé
s0m = A * np.sin(2 * np.pi * nu0 * tm_bit)
s1m = A * np.sin(2 * np.pi * nu1 * tm_bit)

sm = np.concatenate([s0m if bit==0 else s1m for bit in sb])
# vecteur temps du signal modulé
Nm = len(sm)
tm = np.linspace(0, Nm/F_mod, Nm, endpoint=False)

In [291]:
Nbits_to_plot = 16
Npts_to_plot = Nbits_to_plot * N_mod

# Affichage de s_m
df_sm = pd.DataFrame({'x': tm[:Npts_to_plot] , 'y': sm[:Npts_to_plot] , 'legende': 'Signal Module' })
fig3 = px.line(df_sm, x='x',  y='y', color='legende', labels={'x': 'Temps (s)' , 'y':'Signal Module'}, title= 'Modulation du Signal' , width=700 )
fig3.show()

### Affichages

In [292]:
# Affichage de s_b et s_m sur une seule figure
fig4 = make_subplots(specs=[[{"secondary_y": True}]])

fig41 = px.line(df_sm, x='x',  y='y')
    
fig42 = px.scatter(df_sb, x='x', y='y')
fig42.update_traces(yaxis='y2', marker_color='#cc0000', marker_size=8)

fig4.add_traces(fig41.data + fig42.data)
fig4.update_layout(title='Signaux temporels', width=700,
                    xaxis =dict(title=dict(text='Temps (s)')), 
                    yaxis =dict(title=dict(text='signal modulé sm')), 
                    yaxis2=dict(title=dict(text='signal binaire sb',font=dict(color="#cc0000")), color='#cc0000'),
)
fig4.show()

## Séance 2 - Estimation de la Densité De Probabilité (DDP)

### Analyses _in-silico_

In [293]:
## Fonction de calcul d'histogramme
def histo(x, N=None, M=None):
    x = np.asarray(x).ravel()
    
    if N is not None:
        x = x[:N]
    n = len(x)
    
    # Si nombre de classes M non spécifié, utiliser Freedman-Diaconis
    if M is None:
        q75, q25 = np.percentile(x, [75, 25])
        iqr = q75 - q25
        if iqr == 0:
            sigma = np.std(x, ddof=1)
            Delta = 3.5*sigma/n**(1/3)
        else:
            Delta = 2*iqr/n**(1/3)
        M = max(1, int(np.ceil((x.max() - x.min()) / Delta)))
    else:
        Delta = (x.max() - x.min())/M

    # edges et centroïdes
    edges = np.linspace(x.min(), x.max(), M+1)
    c = (edges[:-1] + edges[1:])/2

    # histogramme non normalisé
    counts, _ = np.histogram(x, bins=edges, density=False)
    
    # histogramme normalisé
    DDPest = counts / (n * Delta)
    
    return DDPest, c, Delta

## Foncton d'affichage
def compareDDP(c, DDPest, DDPth, std_estim, title=None):
    # préparer DataFrame pour DDP théorique et bandes ± std
    df_DDP_th                 = pd.DataFrame({'x': c, 'y': DDPth,    'legende':'DDP_th'})
    df_DDP_th_plus_std_estim  = pd.DataFrame({'x': c, 'y': DDPth+std_estim, 'legende':'DDP_th+std'})
    df_DDP_th_minus_std_estim = pd.DataFrame({'x': c, 'y': DDPth-std_estim, 'legende':'DDP_th-std'})
    
    df_DDP_th = pd.concat([df_DDP_th, df_DDP_th_plus_std_estim, df_DDP_th_minus_std_estim])
    
    fig = px.line(df_DDP_th, x='x',  y='y', title=title,
                  color='legende', color_discrete_sequence=['orangered','black','black'], 
                  line_dash='legende', line_dash_sequence=['solid', 'dash', 'dash'])
    fig.add_bar(x=c, y=DDPest, marker_color='gray', name="DDP estimée")
    fig.show()

In [294]:
# analyse du canal test
b = tsa.canal_test()  # récupération du signal du canal test
n_samples = 5000
b = tsa.canal_test(filtered=False, init=0)
DDPth, c_th, Delta_th = histo(b, N=len(b), M=500)

#### Influence de N

In [295]:
M_fixed = 20
N_values = [100, 500, 1000, 5000]
results_N = []

for N in N_values:
    DDPest, c, Delta = histo(b, N=N, M=M_fixed)
    
    # Interpolation de la DDP "théorique" sur les centroïdes actuels
    DDPth_interp = np.interp(c, c_th, DDPth)
    
    biais = np.mean(DDPest - DDPth_interp)
    var = np.var(DDPest)
    
    results_N.append({'N': N, 'biais': biais, 'variance': var})

df_N = pd.DataFrame(results_N)
print("Effet de N sur la qualité de l'estimation")
print(df_N)

Effet de N sur la qualité de l'estimation
      N     biais  variance
0   100  0.000856  0.016787
1   500  0.000206  0.018218
2  1000  0.001235  0.019759
3  5000 -0.002226  0.019967


#### Influence de $\Delta$

In [296]:
N_fixed = 1000
M_values = [5, 20, 50, 100]
results_M = []

for M in M_values:
    DDPest, c, Delta = histo(b, N=N_fixed, M=M)
    DDPth_interp = np.interp(c, c_th, DDPth)
    
    biais = np.mean(DDPest - DDPth_interp)
    var = np.var(DDPest)
    
    results_M.append({'M': M, 'Delta': Delta, 'biais': biais, 'variance': var})

df_M = pd.DataFrame(results_M)
print("Effet de Δ sur la qualité de l'estimation")
print(df_M)

Effet de Δ sur la qualité de l'estimation
     M     Delta     biais  variance
0    5  1.163409 -0.002348  0.017548
1   20  0.290852  0.001235  0.019759
2   50  0.116341  0.001606  0.020589
3  100  0.058170  0.000534  0.021692


### Analyses _in-situ_

In [297]:
# analyse du canal de transmission réel
canal_id = 1        # identifiant du canal à utiliser
powB = 1            # puissance du bruit (paramètre optionnel)
sm_tr = tsa.transmit(sm, canal_id, powB)  # signal modulé transmis

bruit = sm_tr - sm
moyenne = np.mean(bruit)
variance = np.var(bruit)
ecart_type = np.std(bruit)
puissance = np.mean(bruit**2)

print("Caractérisation statistique du canal de transmission")
print(f"Moyenne du bruit: {moyenne:.4f}")
print(f"Variance du bruit: {variance:.4f}")
print(f"Ecart-type du bruit: {ecart_type:.4f}")
print(f"Puissance du bruit: {puissance:.4f}")

DDPbruit, c_bruit, Delta_bruit = histo(bruit, N=len(bruit), M=50)

df_bruit = pd.DataFrame({'x': c_bruit,'y': DDPbruit,'legende': 'DDP bruit'})
fig = px.bar(df_bruit, x='x', y='y', color='legende', labels={'x':'Amplitude du bruit', 'y':'Densité de probabilité'}, title='DDP du bruit ajouté par le canal', width=700)
fig.show()

Caractérisation statistique du canal de transmission
Moyenne du bruit: -0.0000
Variance du bruit: 0.9994
Ecart-type du bruit: 0.9997
Puissance du bruit: 0.9994


## Séance 3 - Estimation de la Densité Spectrale de Puissance Moyenne

### Analyses _in-silico_

In [298]:
def compareDSP(f,DSPth,DSPbiais,DSPest):
    """Fonction d’affichage compareDSP(f, DSPth, DSPbiais, DSPest) commune aux trois estimateurs, qui trace en dB sur un même graphe.
    Args:
        f (_type_): vecteur de fréquences réduites tel que 0 ≤ f < 0.5
        DSPth (_type_): la DSPM théorique vraie du canal test ΓX (f),
        DSPbiais (_type_): la DSPM théorique obtenue en utilisant l’estimateur ??? du canal test ΓX ∗W_{N,B}(f),
        DSPest (_type_): votre DSPM estimée dΓ_{???}(f),
    """
    
    pass

##### Estimateur spectral simple

In [299]:
def estimateur_simple(x, nd, N, nfft):
    pass
    return DSPest1,f

##### Estimateur spectral moyenné

In [300]:
def estimateur_moyenne(x, M, nfft):
    pass
    #signal.welch()
    return DSPest2,f

##### Estimateur spectral de Welch

In [301]:
def estimateur_welch(x, window, M, Noverlap, nfft):
    pass
    #signal.welch()
    return DSPest3,f

### Analyses _in-situ_

In [302]:
...

Ellipsis

## Séance 4 - Détection d’un signal noyé dans un bruit

In [303]:
# Transmission
sm_tr = tsa.transmit(sm, canal_id)

#### Débruitage par détection

In [304]:
th =               # vecteur temps d'un symbole

# reponses impulsionnelles des filtres de détection
h0 = 
h1 = 

# filtrage en Fourier
Nfft = 

fm =        # vecteur de fréquences réduites
...

# signaux en sortie des 2 filtres
s0 = 
s1 =

# signal détecté
sd = ...
sd = sd.astype(int)


SyntaxError: invalid syntax (2641988041.py, line 1)

#### Étude de l’effet du bruit de transmission sur la précision

In [ ]:
def error(sd,sb):
    pass
    return e



#### Décodage

In [ ]:
s_tr = 

## Séance 5 - Prédiction AR d’ordre $\textit{M}$

#### Estimation de la fonction d’autocorrélation

In [ ]:
K = ...

Gam = signal.correlate(s_tr.astype(np.float32), s_tr.astype(np.float32), mode='full', method='auto')
Gam /= len(s_tr)
Gam = Gam[len(Gam)//2 - K:len(Gam)//2 + K + 1]
lags = signal.correlation_lags(len(s_tr), len(s_tr), mode="full")
lags = lags[len(lags)//2 - K:len(lags)//2 + K + 1]

#### Identification du modèle AR(M)

In [ ]:
M = 

# matrice du systeme lineaire 
G = toeplitz()

# second membre du systeme lineaire
b = 

# solution du systeme G.Phi = b
Phi = np.linalg.pinv(G) @ b

# coefficients du filtre de Wiener
h = 

# puissance de l'erreur
sigma = 

#### Prédiction Linéaire

In [ ]:
s_hat =

#### Restauration par filtrage causal/anti-causal

In [ ]:
...